# 3.5 Email Agent — DJEF Email Assistant 🇩🇿✉️

**by Kassouar Fatima 🇩🇿 — Module 3 — Production-Ready Agent**

**DJEF** = *Djazair Email Facilitator* — a warm Algerian email assistant that
authenticates, reads the inbox and sends mail, with a **human approving every send**.

## Goals

1. **Context** carries the credentials (`EmailContext`) — never the state.
2. **Custom state** (`AuthenticatedState`) records whether we are authenticated.
3. **Tools** — `authenticate`, `check_inbox`, `send_email`.
4. **Dynamic tools** — before login *only* `authenticate` exists: the door.
5. **Dynamic prompt** — doorman → DJEF once authenticated.
6. **Human-in-the-Loop** — `send_email` interrupts for approve / edit / reject.

> This notebook is the **lab version of the Streamlit app in this same folder**:
> `app.py` · `components.py` · `design.py` · `runner.py` · `email_agent_service.py`.
> Same tools, same middleware, same HITL decisions — just without the widgets.

## Marhaba! ✉️🇩🇿

Yalla! Before the code, the vocabulary DJEF uses naturally:

| | |
|---|---|
| **Marhaba!** | hello |
| **Khoya** | my brother (friendly) |
| **Saha** | to your health / thank you |
| **Mabrouk!** | congratulations |
| **Barak Allahu fik** | may God bless you |
| **Yalla!** | let's go |
| **Bienvenue!** | welcome |
| **بيكي خير** | may your day be good |

Two notes before we start:

- Credentials go in **context**, the login verdict goes in **state**.
- Nothing here ever opens a mail socket unless the credentials are real
  (see *Marhaba, the IMAP bug* further down 📬).

### 1 · Imports and keys 🔑

`load_dotenv()` reads the nearest `.env` — the Gemini key never lives in code.

In [1]:
from __future__ import annotations

import asyncio
import os
import pprint
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

from dotenv import dotenv_values, load_dotenv
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import (
    HumanInTheLoopMiddleware,
    dynamic_prompt,
    wrap_model_call,
)
from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


def find_env_file() -> Path | None:
    """The nearest .env: this folder first, then every parent."""
    here = Path.cwd()
    for directory in [here, *here.parents]:
        for candidate in (directory / ".env", directory / "streamlit" / ".env"):
            if candidate.is_file():
                return candidate
    return None


ENV_FILE = find_env_file()
if ENV_FILE:
    load_dotenv(ENV_FILE, override=True)
    print(f"  .env   -> {ENV_FILE}")
else:
    print("  .env   -> NOT FOUND (set GOOGLE_API_KEY in the environment)")

_api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY") or ""
print(f"  Gemini -> {'set (len=' + str(len(_api_key)) + ')' if _api_key else 'MISSING'}")

MODEL = "google_genai:gemini-3.5-flash-lite"

  .env   -> C:\Users\Kassouar Fatima\Downloads\LangChain\Module 3 - Production-Ready Agent\chatui_emailassistent\streamlit\.env
  Gemini -> set (len=53)


### 2 · Context and state 🧳

**Context** (`EmailContext`) travels with every call and is what `authenticate`
checks against. **State** (`AuthenticatedState`) is what the middleware reads to
decide whether the door is open. Splitting them is the whole point of Module 3.

In [2]:
@dataclass
class EmailContext:
    """Credentials held in context (never state)."""

    email_address: str = "kassouarfatima@gmail.com"
    password: str = "password123"
    display_name: str = "Kassouar Fatima"


class AuthenticatedState(AgentState):
    """State extended with the authentication verdict."""

    authenticated: bool


print("  EmailContext      ->", EmailContext().email_address)
print("  AuthenticatedState ->", AuthenticatedState.__annotations__.keys())

  EmailContext      -> kassouarfatima@gmail.com
  AuthenticatedState -> dict_keys(['messages', 'jump_to', 'structured_response', 'authenticated'])


### 3 · The inbox 📬

A tiny in-memory store, seeded with three messages so `check_inbox` has
something to show on the very first run.

In [3]:
_email_store: list[dict[str, Any]] = []


def _now() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def _seed_emails() -> None:
    if not _email_store:
        _email_store.extend([
            {"id": 1, "from": "fatima@example.com",
             "to": "kassouarfatima@gmail.com", "subject": "Coffee invitation",
             "body": "Hi! I'm going to be in town next week and was wondering "
                     "if we could grab a coffee?\n- best, Fatima",
             "time": _now(), "type": "received"},
            {"id": 2, "from": "kassouarfatima@gmail.com",
             "to": "fatima@example.com", "subject": "Re: Coffee",
             "body": "I'll be there, bring the coffee!",
             "time": _now(), "type": "sent"},
            {"id": 3, "from": "boss@example.com",
             "to": "kassouarfatima@gmail.com", "subject": "Project deadline",
             "body": "Hi Kassouar, the project deadline is next Friday. "
                     "Please confirm.\n- Boss",
             "time": _now(), "type": "received"},
        ])


_seed_emails()
print(f"  seeded {_email_store and len(_email_store)} emails")

  seeded 3 emails


### 4 · Marhaba, the IMAP bug 🛠️

The raw version of this project called Gmail like this:

```python
from_pw = os.getenv("EMAIL_PASSWORD", "")   # "" when the key is missing
mail.login(from_addr, from_pw)              # blank password
```

Gmail's reply, passed straight to the chat:

```
❌ IMAP error! 🇩🇿
Saha, something went wrong: b'Empty username or password. 5b1f17b1804b1-49ff9072710mb594939845e9'.
```

**The fix:** never open a socket unless the password is a real one. Settings are
read from `.env`, empty counts as *unset*, and if there is no App Password the
tools use the in-memory store above instead. `MAIL_MODE` forces one or the other.

This mirrors `email_agent_service.py` in the app — `setting()`, `mail_config()`,
`fetch_inbox()` and `deliver_email()`.

In [4]:
DEMO_PASSWORDS = {
    "", "password123", "changeme", "change-me", "change_me",
    "your_app_password", "your-app-password", "yourappassword",
    "your_password", "your-password", "xxx", "none", "null", "todo",
}


def setting(name: str, default: str = "") -> str:
    """Read a setting: .env first, then the environment. Empty means unset."""
    values = dotenv_values(ENV_FILE) if ENV_FILE else {}
    value = values.get(name) or os.getenv(name) or ""
    value = str(value).strip()
    if len(value) >= 2 and value[0] == value[-1] and value[0] in "\"'":
        value = value[1:-1].strip()
    return value if value else default


def mail_config() -> dict[str, Any]:
    """Resolve the transport, always with an explicit reason."""
    requested = setting("MAIL_MODE", "auto").lower()
    if requested not in ("auto", "real", "simulated"):
        requested = "auto"

    password = setting("EMAIL_PASSWORD")
    reason = ""
    if requested == "simulated":
        reason = "MAIL_MODE=simulated in .env"
    elif not password:
        reason = "EMAIL_PASSWORD is not set in .env"
    elif password.lower() in DEMO_PASSWORDS:
        reason = (f"EMAIL_PASSWORD is the demo value '{password}' — "
                  "Gmail needs a 16-character App Password")

    mode = "simulated" if reason else "real"
    if requested == "real" and reason:
        mode = "real"                      # asked for it: report it, don't hide it

    usable = bool(mode == "real" and not reason
                  and password.strip()
                  and password.lower() not in DEMO_PASSWORDS)

    return {"mode": mode, "requested": requested, "usable": usable,
            "forced": requested == "real" and bool(reason),
            "address": setting("EMAIL_ADDRESS", "kassouarfatima@gmail.com"),
            "password": password, "reason": reason,
            "smtp_host": setting("SMTP_HOST", "smtp.gmail.com"),
            "imap_host": setting("IMAP_HOST", "imap.gmail.com")}


MAIL = mail_config()
print(f"  MAIL_MODE  {MAIL['requested']} -> {MAIL['mode']}")
print(f"  usable?    {MAIL['usable']}")
print(f"  reason     {MAIL['reason'] or '(none — real mail configured)'}")

  MAIL_MODE  auto -> simulated
  usable?    False
  reason     EMAIL_PASSWORD is not set in .env


### 5 · The two dispatchers 🚦

`fetch_inbox()` and `deliver_email()` are what the tools actually call. They
pick the real transport only when `MAIL["usable"]` is true, otherwise the store.

In [5]:
def receive_email_simulated() -> str:
    """All emails from the in-memory store."""
    if not _email_store:
        return "📭 **Your inbox is empty, Khoya!** 📭\n\nTry sending an email first! ✉️🇩🇿"
    lines = ["📬 **Your Inbox — Marhaba!** 🇩🇿📬\n"]
    for mail in reversed(_email_store):
        lines.append(
            f"📥 [{mail['time']}] {mail['from']}\n"
            f"   **To:** {mail['to']}\n"
            f"   **Subject:** {mail['subject']}\n"
            f"   **Type:** {mail['type']}\n"
            f"   {mail['body'][:150]}{'…' if len(mail['body']) > 150 else ''}\n"
        )
    return "\n".join(lines)


def send_email_simulated(to: str, subject: str, body: str) -> str:
    """Store an email in memory and confirm."""
    _email_store.append({"id": len(_email_store) + 1,
                         "from": MAIL["address"], "to": to, "subject": subject,
                         "body": body, "time": _now(), "type": "sent"})
    return (
        f"✅ **Email sent!** 🇩🇿✉️\n\n"
        f"**Marhaba!** Your email has been sent successfully!\n"
        f"**To:** {to}\n**Subject:** {subject}\n**Body:** {body}\n"
        f"**Sent at:** {_now()}\n**From:** {MAIL['address']}\n\n"
        f"🥳 **Mabrouk!** Message delivered! 🇩🇿\n"
        f"**Barak Allahu fik!** (God bless you) ✉️🌟"
    )


def _config_error(protocol: str) -> str:
    """Actionable message naming the exact .env key — never a raw server reply."""
    return (
        f"❌ **{protocol} not configured** 🇩🇿\n\n"
        f"**Saha, Khoya** — {MAIL['reason']}.\n\n"
        f"Set `EMAIL_PASSWORD` in `.env` to a 16-character Gmail App Password:\n"
        f"https://myaccount.google.com/apppasswords\n\n"
        f"*Or leave it empty and keep the built-in demo inbox — it works offline.* 🇩🇿"
    )


def fetch_inbox() -> str:
    """Real IMAP when usable, the in-memory store otherwise."""
    if MAIL["usable"]:
        return _config_error("IMAP")   # real fetch would go here (see the app)
    if MAIL["forced"]:
        return _config_error("IMAP")
    return receive_email_simulated()


def deliver_email(to: str, subject: str, body: str) -> str:
    """Real SMTP when usable, the in-memory store otherwise."""
    if MAIL["usable"]:
        return _config_error("SMTP")   # real send would go here (see the app)
    if MAIL["forced"]:
        return _config_error("SMTP")
    return send_email_simulated(to, subject, body)


print(fetch_inbox()[:200])

📬 **Your Inbox — Marhaba!** 🇩🇿📬

📥 [2026-09-27 19:11:32] boss@example.com
   **To:** kassouarfatima@gmail.com
   **Subject:** Project deadline
   **Type:** received
   Hi Kassouar, the project deadlin


### 6 · The three tools 🛠️

`authenticate` returns a `Command` — it writes `authenticated` into **state**
and answers the model with a `ToolMessage` in the same breath.

In [6]:
@tool
def check_inbox() -> str:
    """Check all emails in your inbox — received and sent, with timestamps."""
    return fetch_inbox()


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to any address."""
    return deliver_email(to, subject, body)


@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password."""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(
            update={
                "authenticated": True,
                "messages": [
                    ToolMessage(
                        "Marhaba! Successfully authenticated! Welcome to DJEF. ✉️🇩🇿",
                        tool_call_id=runtime.tool_call_id,
                    )
                ],
            }
        )
    return Command(
        update={
            "authenticated": False,
            "messages": [
                ToolMessage(
                    "Authentication failed. Check your email and password.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )


print("  tools ->", [t.name for t in (authenticate, check_inbox, send_email)])

  tools -> ['authenticate', 'check_inbox', 'send_email']


### 7 · The door — dynamic tools 🔒

One `wrap_model_call` middleware swaps the tool list on every turn. Before
login the model can only see `authenticate`, so it is physically unable to read
the inbox — the door is the tool list, not a sentence in the prompt.

In [7]:
@wrap_model_call
async def dynamic_tool_call(request: Any, handler: Any) -> Any:
    """Allow the full email toolbox only after authentication."""
    if request.state.get("authenticated"):
        request = request.override(tools=[authenticate, check_inbox, send_email])
    else:
        request = request.override(tools=[authenticate])
    return await handler(request)


print("  middleware -> dynamic_tool_call")

  middleware -> dynamic_tool_call


### 8 · The voice — dynamic prompt 💬

The doorman before login, DJEF after. Same model, different personality.

In [8]:
authenticated_prompt = (
    "You are DJEF (Djazair Email Facilitator) — a full-featured Algerian email assistant. "
    "You have a warm, friendly Algerian personality. Use phrases like 'Marhaba!', "
    "'Khoya!', 'Saha!', 'Mabrouk!', 'Barak Allahu fik!', 'Bienvenue!' naturally. "
    "You can check the inbox and send emails. "
    "Always show timestamps for every email, and the sender and receiver. "
    "Your first step after authentication is to check the inbox. "
    "Be helpful, polite, and proudly Algerian. 🇩🇿"
)

unauthenticated_prompt = (
    "You are a friendly Algerian doorman assistant. Greet with 'Marhaba! ✉️🇩🇿' "
    "and ask the visitor for their email and password so you can let them in. "
    "Be warm, welcoming, and speak with Algerian charm. "
    "You cannot read or send mail yet — say so politely and ask for the credentials."
)


@dynamic_prompt
def dynamic_prompt_fn(request: Any) -> str:
    """Doorman before authentication, DJEF after."""
    return authenticated_prompt if request.state.get("authenticated") else unauthenticated_prompt


print("  middleware -> dynamic_prompt")

  middleware -> dynamic_prompt


### 9 · Building the agent 🏗️

`InMemorySaver` keeps the thread alive so `Command(resume=...)` can continue
exactly where the interrupt left off. `HumanInTheLoopMiddleware` puts the door
in front of `send_email` only — `authenticate` and `check_inbox` run free.

In [9]:
checkpointer = InMemorySaver()

agent = create_agent(
    MODEL,
    tools=[authenticate, check_inbox, send_email],
    checkpointer=checkpointer,
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call,
        dynamic_prompt_fn,
        HumanInTheLoopMiddleware(
            interrupt_on={"authenticate": False, "check_inbox": False, "send_email": True}
        ),
    ],
)

print(f"  agent built -> {MODEL}")

  agent built -> google_genai:gemini-3.5-flash-lite


### 10 · Run helpers ▶️

Three small wrappers: send a message, resume a decision, and pull the pending
draft body out of an interrupt.

> **Why `await`, and not `asyncio.run(...)`?**
> A Jupyter kernel already has a running event loop, so calling the stdlib
> runner from a cell raises
> `RuntimeError: asyncio.run() cannot be called from a running event loop`.
> The cells below therefore use **top-level `await`**, which ipykernel supports
> natively. The Streamlit app (`runner.py`) keeps the stdlib runner because it
> runs *outside* any loop.

In [10]:
async def run(message: str, thread_id: str = "djef") -> dict:
    """Send one message on a thread (context is fresh each call)."""
    return await agent.ainvoke(
        {"messages": [HumanMessage(content=message.strip())]},
        context=EmailContext(),
        config={"configurable": {"thread_id": thread_id}},
    )


async def resume(decision: str, thread_id: str = "djef",
                 edited_body: str = "", name: str = "send_email") -> dict:
    """Continue after an interrupt: approve, reject or edit."""
    if decision == "edit":
        body = dict(
            # keep every original arg (to/subject) and only swap the body
            await _pending_args(thread_id)
        )
        body["body"] = edited_body
        payload = {"decisions": [{"type": "edit",
                                  "edited_action": {"name": name, "args": body}}]}
    else:
        payload = {"decisions": [{"type": decision}]}
    return await agent.ainvoke(
        Command(resume=payload),
        context=EmailContext(),
        config={"configurable": {"thread_id": thread_id}},
    )


async def _pending_args(thread_id: str) -> dict[str, Any]:
    """Read the interrupted call's original args off the paused checkpoint."""
    try:
        snapshot = await agent.aget_state({"configurable": {"thread_id": thread_id}})
        value = snapshot.tasks[0].interrupts[0].value
        return dict(value["action_requests"][0]["args"] or {})
    except (AttributeError, IndexError, KeyError, TypeError):
        return {}


def interrupt_body(response: dict) -> str | None:
    """The pending draft body, or None when nothing interrupted."""
    try:
        interrupt = response["__interrupt__"][0]
        value = interrupt.value if hasattr(interrupt, "value") else interrupt["value"]
        return value["action_requests"][0]["args"]["body"]
    except (KeyError, IndexError, TypeError, AttributeError):
        return None


def last_text(response: dict) -> str:
    """The last human-readable text in the response."""
    for msg in reversed(response.get("messages", [])):
        content = getattr(msg, "content", None)
        if isinstance(content, str) and content:
            return content
        if isinstance(content, list):
            text = " ".join(str(c) for c in content if isinstance(c, str) and c)
            if text:
                return text
    return ""


print("  helpers ready")

  helpers ready


### 11 · Step 1 — authenticate 🔐

First message carries the credentials. The model can *only* call
`authenticate`, so this is the only door it can knock on.

In [11]:
response = await run("kassouarfatima@gmail.com, password123", "djef")
print(last_text(response))

📬 **Your Inbox — Marhaba!** 🇩🇿📬

📥 [2026-09-27 19:11:32] boss@example.com
   **To:** kassouarfatima@gmail.com
   **Subject:** Project deadline
   **Type:** received
   Hi Kassouar, the project deadline is next Friday. Please confirm.
- Boss

📥 [2026-09-27 19:11:32] kassouarfatima@gmail.com
   **To:** fatima@example.com
   **Subject:** Re: Coffee
   **Type:** sent
   I'll be there, bring the coffee!

📥 [2026-09-27 19:11:32] fatima@example.com
   **To:** kassouarfatima@gmail.com
   **Subject:** Coffee invitation
   **Type:** received
   Hi! I'm going to be in town next week and was wondering if we could grab a coffee?
- best, Fatima



### 12 · Step 2 — check the inbox 📬

Now the toolbox opens: `check_inbox` and `send_email` join `authenticate`.

In [12]:
response = await run("please check my inbox", "djef")
print(last_text(response))

📬 **Your Inbox — Marhaba!** 🇩🇿📬

📥 [2026-09-27 19:11:32] boss@example.com
   **To:** kassouarfatima@gmail.com
   **Subject:** Project deadline
   **Type:** received
   Hi Kassouar, the project deadline is next Friday. Please confirm.
- Boss

📥 [2026-09-27 19:11:32] kassouarfatima@gmail.com
   **To:** fatima@example.com
   **Subject:** Re: Coffee
   **Type:** sent
   I'll be there, bring the coffee!

📥 [2026-09-27 19:11:32] fatima@example.com
   **To:** kassouarfatima@gmail.com
   **Subject:** Coffee invitation
   **Type:** received
   Hi! I'm going to be in town next week and was wondering if we could grab a coffee?
- best, Fatima



### 13 · Step 3 — send, and the door holds ✉️⏸️

`send_email` is registered with `interrupt_on=True`, so the graph stops and
hands us the draft. Nothing is sent until a human says so.

In [13]:
response = await run(
    "send an email to khoya@example.com with subject 'Saha!' "
    "and body 'Marhaba Khoya, the coffee is on me. Mabrouk! - Kassouar'",
    "djef",
)

pending = interrupt_body(response)
if pending:
    print("⏸️  HITL interrupt — pending draft:\n")
    print(pending)
else:
    print("  (no interrupt raised — the model answered in prose)")
    print(" ", last_text(response))

⏸️  HITL interrupt — pending draft:

Marhaba Khoya, the coffee is on me. Mabrouk! - Kassouar


### 14 · Step 4 — approve ✅

`Command(resume={"decisions": [{"type": "approve"}]})` continues the same thread
as if nothing happened — the graph picks up exactly where it paused.

In [14]:
if pending:
    response = await resume("approve", "djef")
    print(last_text(response))
else:
    print("  skipped — no interrupt to approve")

✅ **Email sent!** 🇩🇿✉️

**Marhaba!** Your email has been sent successfully!
**To:** khoya@example.com
**Subject:** Saha!
**Body:** Marhaba Khoya, the coffee is on me. Mabrouk! - Kassouar
**Sent at:** 2026-09-27 19:11:41
**From:** kassouarfatima@gmail.com

🥳 **Mabrouk!** Message delivered! 🇩🇿
**Barak Allahu fik!** (God bless you) ✉️🌟


### 15 · Step 5 — reject ❌

A reject carries a message the model must obey. Ask it to leave the draft alone
and it should stop trying to send.

In [15]:
response = await run(
    "now send another email to boss@example.com with subject 'Deadline' "
    "and body 'The deadline is fine, Khoya.'",
    "djef",
)
pending = interrupt_body(response)
if pending:
    print("⏸️  pending:", pending[:120])
    response = await resume(
        "reject", "djef",
        # a reject may carry a message the model has to obey
    )
    print("\n-> after REJECT:\n", last_text(response))
else:
    print("  (no interrupt raised — the model answered in prose)")
    print(" ", last_text(response))

⏸️  pending: The deadline is fine, Khoya.



-> after REJECT:
 User rejected the tool call for `send_email` with id call_102286. The tool was not executed. Do not retry this tool call unless the user explicitly requests it.


### 16 · Step 6 — edit before it goes ✏️

`"type": "edit"` swaps in `edited_action`. **Keep the original `args`** and
only replace `body` — sending a fresh `{"body": ...}` alone would drop `to` and
`subject`, and the tool would fail validation. (That exact bug lived in
`runner.py`'s resume path; see `email_agent_service.resume_email_edit`.)

In [16]:
response = await run(
    "send an email to fatima@example.com with subject 'Coffee' "
    "and body 'See you at 5.'",
    "djef",
)
pending = interrupt_body(response)
if pending:
    print("⏸️  original draft:", pending)
    edited = "Marhaba Fatima! See you at 5, insha'Allah. Saha! - Kassouar 🇩🇿"
    response = await resume("edit", "djef", edited_body=edited)
    print("\n-> after EDIT:\n", last_text(response))
else:
    print("  (no interrupt raised — the model answered in prose)")
    print(" ", last_text(response))

⏸️  original draft: See you at 5.



-> after EDIT:
 Note: a human reviewer replaced this tool call before it ran. The call recorded in your message is the one you produced, not the one that executed. This was intentional and authorized. Do not re-issue your original call. Executed instead: send_email with arguments {"to": "fatima@example.com", "subject": "Coffee", "body": "Marhaba Fatima! See you at 5, insha'Allah. Saha! - Kassouar \ud83c\udde9\ud83c\uddff"}.

Tool response:
✅ **Email sent!** 🇩🇿✉️

**Marhaba!** Your email has been sent successfully!
**To:** fatima@example.com
**Subject:** Coffee
**Body:** Marhaba Fatima! See you at 5, insha'Allah. Saha! - Kassouar 🇩🇿
**Sent at:** 2026-09-27 19:11:45
**From:** kassouarfatima@gmail.com

🥳 **Mabrouk!** Message delivered! 🇩🇿
**Barak Allahu fik!** (God bless you) ✉️🌟


### 17 · The full state 🔍

Everything the thread accumulated, straight from the checkpointer.

In [17]:
final = await run("thanks Khoya, that is all for today", "djef")
print(last_text(final))
print()
pprint.pprint(final)

thanks Khoya, that is all for today

{'authenticated': True,
 'messages': [HumanMessage(content='kassouarfatima@gmail.com, password123', additional_kwargs={}, response_metadata={}, id='e8950d11-a63a-4e93-b83c-98700731f731'),
              AIMessage(content=[{'type': 'text', 'text': "Marhaba! ✉️🇩🇿 \n\nWelcome! Ahlan wa sahlan! It is an absolute pleasure to have you here today. \n\nBefore I can let you in, I just need your credentials—if you could kindly provide your email and password, that would be wonderful. \n\n*(By the way, as your loyal doorman, I must mention that I cannot read or send mail yet, but I can certainly take your login details to get you through the door!)* \n\nLet me just process what you've given me right now..."}], additional_kwargs={'function_call': {'name': 'authenticate', 'arguments': '{"password": "password123", "email": "kassouarfatima@gmail.com"}'}, '__gemini_function_call_thought_signatures__': {'call_184641': 'EmAKXgFpFH0TcE9L/9gp0tEJg3jSo+EJN/Tx9EfJ68oEZdCW

---

## Mabrouk! 🇩🇿

**Mabrouk! 🇩🇿 — by Kassouar Fatima**

What you just ran:

| # | Pattern | Where it lives in the app |
|---|---|---|
| 1 | Context vs state | `EmailContext`, `AuthenticatedState` |
| 2 | Credentials never in state | `runner.build_context()` |
| 3 | Dynamic tools (the door) | `dynamic_tool_call` middleware |
| 4 | Dynamic prompt (the voice) | `dynamic_prompt_fn` middleware |
| 5 | HITL approve / reject / edit | `resume_email_approval`, `resume_email_edit` |
| 6 | Credentials read safely from `.env` | `setting()`, `mail_config()` |

**Barak Allahu fik!** ✉️🇩🇿 · *Yalla!*